<h2>Import requirements

In [2]:
import time
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")
import tensorflow as tf
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
seed = 0; np.random.seed(seed)

<h1>>> Regular data

<h2>Import and split data

In [4]:
DATA = np.loadtxt('data/league_matches_dataset.csv', delimiter=',', skiprows=1)
X = DATA[:, :-1]
y = DATA[:, -1]
X_train, X_other, y_train, y_other = train_test_split(X, y, train_size=0.7, random_state=seed)
X_val, X_test, y_val, y_test = train_test_split(X_other, y_other, test_size=0.5, random_state=seed)

<h2>Hyperparameter tuning

<h4>Define dummy model for tuning

In [11]:
def test_model(X_train, y_train, X_val, y_val, units_per_layer, dropout_rate, learning_rate, batch_size, num_epochs):
    first, second, third = units_per_layer

    if dropout_rate:
        model = tf.keras.Sequential([
            tf.keras.layers.Dense(first, activation='relu'),
            tf.keras.layers.Dropout(dropout_rate),
            tf.keras.layers.Dense(second, activation='relu'),
            tf.keras.layers.Dropout(dropout_rate),
            tf.keras.layers.Dense(third, activation='relu'),
            tf.keras.layers.Dropout(dropout_rate),
            tf.keras.layers.Dense(1, activation='sigmoid')
        ])
    else:
        model = tf.keras.Sequential([
            tf.keras.layers.Dense(first, activation='relu'),
            tf.keras.layers.Dense(second, activation='relu'),
            tf.keras.layers.Dense(third, activation='relu'),
            tf.keras.layers.Dense(1, activation='sigmoid')
        ])

    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    
    model.compile(
        optimizer=optimizer,
        loss=tf.keras.losses.BinaryCrossentropy(),
        metrics=['accuracy']
        )

    history = model.fit(
        X_train, 
        y_train, 
        batch_size=batch_size, 
        epochs=num_epochs, 
        validation_data=(X_val, y_val),
        verbose=0
        )

    print(f"VAL ACCURACY: {round(history.history["val_accuracy"][-1], 5)}  |  units_per_layer: {units_per_layer}, dropout_rate: {dropout_rate}, learning_rate: {learning_rate}, batch_size: {batch_size}, num_epochs: {num_epochs}")

<h4>Test parameters

In [ ]:
layers = [[128, 128, 64], [128, 64, 64], [64, 64, 64], [128, 64, 32], [128, 32, 32]]
dropout_rates = [None, 0.2]
learning_rates = [0.0001, 0.001, 0.01] # i tried also 0.1, 1.0 and 10.0 but these were consistently worse
batch_sizes = [64, 128]
nums_epochs = [5, 10, 15]

for units_per_layer in layers:
    print("\nNEW UNITS PER LAYER: ", units_per_layer)
    for dropout_rate in dropout_rates:
        for learning_rate in learning_rates:
            for batch_size in batch_sizes:
                for num_epochs in nums_epochs:
                    test_model(X_train, y_train, X_val, y_val, units_per_layer, dropout_rate, learning_rate, batch_size, num_epochs)


NEW UNITS PER LAYER:  [128, 128, 64]
VAL ACCURACY: 0.77016  |  units_per_layer: [128, 128, 64], dropout_rate: None, learning_rate: 0.0001, batch_size: 64, num_epochs: 5
VAL ACCURACY: 0.78145  |  units_per_layer: [128, 128, 64], dropout_rate: None, learning_rate: 0.0001, batch_size: 64, num_epochs: 10
VAL ACCURACY: 0.90209  |  units_per_layer: [128, 128, 64], dropout_rate: None, learning_rate: 0.0001, batch_size: 64, num_epochs: 15
VAL ACCURACY: 0.86575  |  units_per_layer: [128, 128, 64], dropout_rate: None, learning_rate: 0.0001, batch_size: 128, num_epochs: 5
VAL ACCURACY: 0.90972  |  units_per_layer: [128, 128, 64], dropout_rate: None, learning_rate: 0.0001, batch_size: 128, num_epochs: 10
VAL ACCURACY: 0.9079  |  units_per_layer: [128, 128, 64], dropout_rate: None, learning_rate: 0.0001, batch_size: 128, num_epochs: 15
VAL ACCURACY: 0.78311  |  units_per_layer: [128, 128, 64], dropout_rate: None, learning_rate: 0.001, batch_size: 64, num_epochs: 5
VAL ACCURACY: 0.90906  |  units_p

<h2>Define and save the regular model

<h4>The best combination - VAL ACCURACY: 0.95652  |  units_per_layer: [128, 32, 32], dropout_rate: None, learning_rate: 0.001, batch_size: 64, num_epochs: 15

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
    
model.compile(
    optimizer=optimizer,
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=['accuracy']
)

history = model.fit(
    X_train, 
    y_train, 
    batch_size=64, 
    epochs=15, 
    validation_data=(X_val, y_val),
    verbose=0
)

model.save('models/lol_model.keras')

<h1>>> Normalized data

<h2>Import and split data

In [15]:
DATA = np.loadtxt('data/league_matches_dataset_normalized.csv', delimiter=',', skiprows=1)
X = DATA[:, :-1]
y = DATA[:, -1]
X_train, X_other, y_train, y_other = train_test_split(X, y, train_size=0.7, random_state=seed)
X_val, X_test, y_val, y_test = train_test_split(X_other, y_other, test_size=0.5, random_state=seed)

<h2>Hyperparameter tuning

In [16]:
layers = [[128, 128, 64], [128, 64, 64], [64, 64, 64], [128, 64, 32], [128, 32, 32]]
dropout_rates = [None, 0.2]
learning_rates = [0.0001, 0.001, 0.01] # i tried also 0.1, 1.0 and 10.0 but these were consistently worse
batch_sizes = [64, 128]
nums_epochs = [5, 10, 15]

for units_per_layer in layers:
    print("\nNEW UNITS PER LAYER: ", units_per_layer)
    for dropout_rate in dropout_rates:
        for learning_rate in learning_rates:
            for batch_size in batch_sizes:
                for num_epochs in nums_epochs:
                    test_model(X_train, y_train, X_val, y_val, units_per_layer, dropout_rate, learning_rate, batch_size, num_epochs)


NEW UNITS PER LAYER:  [128, 128, 64]
VAL ACCURACY: 0.86658  |  units_per_layer: [128, 128, 64], dropout_rate: None, learning_rate: 0.0001, batch_size: 64, num_epochs: 5
VAL ACCURACY: 0.89081  |  units_per_layer: [128, 128, 64], dropout_rate: None, learning_rate: 0.0001, batch_size: 64, num_epochs: 10
VAL ACCURACY: 0.88467  |  units_per_layer: [128, 128, 64], dropout_rate: None, learning_rate: 0.0001, batch_size: 64, num_epochs: 15
VAL ACCURACY: 0.8241  |  units_per_layer: [128, 128, 64], dropout_rate: None, learning_rate: 0.0001, batch_size: 128, num_epochs: 5
VAL ACCURACY: 0.87122  |  units_per_layer: [128, 128, 64], dropout_rate: None, learning_rate: 0.0001, batch_size: 128, num_epochs: 10
VAL ACCURACY: 0.88799  |  units_per_layer: [128, 128, 64], dropout_rate: None, learning_rate: 0.0001, batch_size: 128, num_epochs: 15
VAL ACCURACY: 0.88102  |  units_per_layer: [128, 128, 64], dropout_rate: None, learning_rate: 0.001, batch_size: 64, num_epochs: 5
VAL ACCURACY: 0.91421  |  units_p

<h2>Define and save the normalized model

<h4>The best combination -  VAL ACCURACY: 0.94739  |  units_per_layer: [128, 64, 64], dropout_rate: None, learning_rate: 0.01, batch_size: 64, num_epochs: 15

In [17]:
model = tf.keras.Sequential([
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)
    
model.compile(
    optimizer=optimizer,
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=['accuracy']
)

history = model.fit(
    X_train, 
    y_train, 
    batch_size= 64, 
    epochs=15, 
    validation_data=(X_val, y_val),
    verbose=0
)

model.save('models/lol_model_normalized.keras')